# Logistic regression analysis

Refactored version of the original notebook. The workflow is kept close to the original analysis, but repeated code has been moved into functions with docstrings.

In [ ]:
import sys
from pathlib import Path
from functools import reduce
import datetime

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mtick
import matplotlib.ticker as mticker
import seaborn as sns

from scipy.cluster.hierarchy import linkage, leaves_list
from tableone import TableOne
import statsmodels.formula.api as smf

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "interim"
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from utils import eda_tools as eda
from utils import data_cleaning_tools as dct
from utils import micro_cleaning_tools as mct
from steps.get_temp_crp import get_temperature, get_crp
from steps.get_colonisation import get_colonisation

cfg_path = "../configs/config.yaml"
cfg = dct.load_config(cfg_path)


## Helper functions

In [ ]:
def clean_missing(x):
    """Convert common string encodings of missing/unknown values to np.nan.

    Parameters
    ----------
    x : object
        Input value from a dataframe cell.

    Returns
    -------
    object
        np.nan if the value is a recognised missing/unknown string; otherwise
        the original value.
    """
    if isinstance(x, str):
        x_clean = x.strip().lower()
        if x_clean in ["unknown", "other", "other_unknown", "none", "nan", "", "None"]:
            return np.nan
    return x


def load_analysis_data(path="../data/interim/analysis_df.csv", surgery_only=True):
    """Load the analysis dataset and optionally restrict to surgical patients.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to the analysis dataframe CSV.
    surgery_only : bool, default=True
        If True, keep only rows where `surgery_before_infection == 1`.

    Returns
    -------
    pandas.DataFrame
        Loaded and optionally filtered analysis dataframe.
    """
    analysis_df = pd.read_csv(path)

    if surgery_only:
        analysis_df = analysis_df[analysis_df["surgery_before_infection"] == 1].copy()

    return analysis_df


def get_table1_df(analysis_df, table1_cols, numeric_columns=None):
    """Prepare the dataframe used for Table 1.

    This keeps the same cleaning logic as the original notebook: select columns
    from the config, clean unknown-like values, and coerce selected variables
    to numeric.

    Parameters
    ----------
    analysis_df : pandas.DataFrame
        Main analysis dataframe.
    table1_cols : dict
        Table 1 settings from the config. Must contain a `columns` entry.
    numeric_columns : list[str], optional
        Variables to coerce to numeric.

    Returns
    -------
    pandas.DataFrame
        Cleaned dataframe for TableOne.
    """
    if numeric_columns is None:
        numeric_columns = ["imd_decile", "age_at_admission", "length_of_stay_days", "temp", "crp"]

    table1_df = analysis_df[table1_cols["columns"]].copy()
    table1_df = table1_df.applymap(clean_missing)

    for c in numeric_columns:
        if c in table1_df.columns:
            table1_df[c] = pd.to_numeric(table1_df[c], errors="coerce")

    return table1_df


def create_table1(table1_df, table1_cols, export_path="table1_epidemiology_esbl.tex"):
    """Create and optionally export a TableOne baseline characteristics table.

    Parameters
    ----------
    table1_df : pandas.DataFrame
        Cleaned dataframe for TableOne.
    table1_cols : dict
        Table 1 settings from the config.
    export_path : str or None, default="table1_epidemiology_esbl.tex"
        Where to save the LaTeX table. If None, no file is written.

    Returns
    -------
    tableone.TableOne
        TableOne object ready for display/export.
    """
    table1 = TableOne(
        data=table1_df,
        columns=table1_cols["columns"],
        categorical=table1_cols["categorical"],
        groupby=table1_cols["groupby"],
        nonnormal=table1_cols["nonnormal"],
        pval=True,
        missing=True,
        include_null=False,
    )

    if export_path is not None:
        table1.to_latex(export_path)

    return table1

In [ ]:
def prepare_lr_df(analysis_df, uni_lr_columns):
    """
    Prepare dataframe for univariable logistic regression.

    This function:
    1. Selects candidate LR columns
    2. Removes unknown values
    3. Converts continuous variables to numeric
    4. Encodes binary variables as 0/1 where mappings are provided
    5. Keeps multi-category variables as categorical
    6. Maps the ESBL outcome to 1/0 if needed

    Parameters
    ----------
    analysis_df : pandas.DataFrame
        Main analysis dataframe.

    uni_lr_columns : dict
        Dictionary containing LR variable groups, e.g.
        "candidate_columns", "outcome", "continuous", "binary",
        "categorical", and optionally "binary_mappings".

    Returns
    -------
    pandas.DataFrame
        Cleaned dataframe ready for univariable logistic regression.
    """

    candidate_cols = uni_lr_columns["candidate_columns"]
    outcome = uni_lr_columns["outcome"]

    continuous_vars = uni_lr_columns.get("continuous", [])
    binary_vars = uni_lr_columns.get("binary", [])
    categorical_vars = uni_lr_columns.get("categorical", [])

    binary_mappings = uni_lr_columns.get("binary_mappings", {})

    uni_lr_df = analysis_df[candidate_cols].copy()

    # Remove unknown / unclear categories using your existing helper
    uni_lr_df = dct.remove_unknowns(uni_lr_df)

    # Convert continuous variables to numeric
    for c in continuous_vars:
        if c in uni_lr_df.columns:
            uni_lr_df[c] = pd.to_numeric(
                uni_lr_df[c],
                errors="coerce"
            )

    # Map outcome to 1/0 if stored as text
    if outcome in uni_lr_df.columns:
        if uni_lr_df[outcome].dtype == "object":
            uni_lr_df[outcome] = uni_lr_df[outcome].map({
                "ESBL": 1,
                "non-ESBL": 0
            })

    # Encode binary predictors using provided mappings
    for c in binary_vars:
        if c in uni_lr_df.columns:
            if c in binary_mappings:
                uni_lr_df[c] = uni_lr_df[c].map(binary_mappings[c])
            else:
                # Generic fallback for common binary labels
                uni_lr_df[c] = uni_lr_df[c].replace({
                    "Yes": 1,
                    "No": 0,
                    "yes": 1,
                    "no": 0,
                    True: 1,
                    False: 0
                })

    # Keep multi-category variables as categorical
    for c in categorical_vars:
        if c in uni_lr_df.columns:
            uni_lr_df[c] = uni_lr_df[c].astype("category")

    return uni_lr_df

In [ ]:
def run_univariable_continuous_lr(uni_lr_df, continuous_vars, outcome="esbl_status"):
    """Run univariable logistic regression for continuous predictors.

    Parameters
    ----------
    uni_lr_df : pandas.DataFrame
        Logistic regression dataframe.
    continuous_vars : list[str]
        Continuous predictors to test one at a time.
    outcome : str, default="esbl_status"
        Binary outcome variable coded 1/0.

    Returns
    -------
    pandas.DataFrame
        One row per continuous predictor with OR, 95% CI and p-value.
    """
    results = []

    for var in continuous_vars:
        model_df = uni_lr_df[[outcome, var]].dropna()
        formula = f"{outcome} ~ {var}"

        try:
            model = smf.logit(formula, data=model_df).fit(disp=0)
            coef = model.params[var]
            p = model.pvalues[var]
            ci_low, ci_high = model.conf_int().loc[var]

            results.append({
                "predictor": var,
                "term": var,
                "odds_ratio": np.exp(coef),
                "ci_lower": np.exp(ci_low),
                "ci_upper": np.exp(ci_high),
                "p_value": p,
                "n": len(model_df),
            })

        except Exception as e:
            results.append({
                "predictor": var,
                "term": var,
                "odds_ratio": np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "p_value": np.nan,
                "n": len(model_df),
                "error": str(e),
            })

    return pd.DataFrame(results)


def run_univariable_categorical_lr(
    uni_lr_df,
    categorical_vars,
    outcome="esbl_status",
    reference_levels=None,
):
    """Run univariable logistic regression for categorical predictors.

    Parameters
    ----------
    uni_lr_df : pandas.DataFrame
        Logistic regression dataframe.
    categorical_vars : list[str]
        Categorical predictors to test one at a time.
    outcome : str, default="esbl_status"
        Binary outcome variable coded 1/0.
    reference_levels : dict, optional
        Mapping of variable name to desired reference level.

    Returns
    -------
    pandas.DataFrame
        One row per non-reference category with OR, 95% CI and p-value.
    """
    if reference_levels is None:
        reference_levels = {
            "site": "blood",
            "organism_bug": "escherichia coli",
            "ethnicity_desc": "white",
            "tfc": "general_surgery", 
            "prophylaxis_group" : 'cefuroxime | metronidazole'}

    cat_results = []

    for var in categorical_vars:
        model_df = uni_lr_df[[outcome, var]].dropna()

        if var in reference_levels:
            ref = reference_levels[var]
            formula = f"{outcome} ~ C({var}, Treatment(reference={ref!r}))"
        else:
            formula = f"{outcome} ~ C({var})"

        try:
            model = smf.logit(formula, data=model_df).fit(disp=0)
            conf = model.conf_int()

            for term in model.params.index:
                if term == "Intercept":
                    continue

                coef = model.params[term]
                p = model.pvalues[term]
                ci_low, ci_high = conf.loc[term]
                level = term.split("[T.")[-1].rstrip("]")

                cat_results.append({
                    "predictor": var,
                    "category": level,
                    "odds_ratio": np.exp(coef),
                    "ci_lower": np.exp(ci_low),
                    "ci_upper": np.exp(ci_high),
                    "p_value": p,
                    "n": len(model_df),
                })

        except Exception as e:
            cat_results.append({
                "predictor": var,
                "category": np.nan,
                "odds_ratio": np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
                "p_value": np.nan,
                "n": len(model_df),
                "error": str(e),
            })

    return pd.DataFrame(cat_results)


def select_variables_for_multivariable_lr(results_df, cat_results_df, p_threshold=0.2):
    """Select candidate variables for multivariable LR using a p-value threshold.
    Also add predictors based on clinical judgement even if they don't meet the threshold.

    Parameters
    ----------
    results_df : pandas.DataFrame
        Continuous univariable LR results.
    cat_results_df : pandas.DataFrame
        Categorical univariable LR results.
    p_threshold : float, default=0.2
        P-value threshold used for screening.

    Returns
    -------
    list[str]
        Unique predictor names that met the screening threshold.
    """
    continuous_candidates = list(
        results_df.loc[results_df["p_value"] < p_threshold, "predictor"]
    )
    categorical_candidates = list(
        cat_results_df.loc[cat_results_df["p_value"] < p_threshold, "predictor"]
    )

    clinically_important_predictors = [
        "age_at_admission",
        "gender", 
    'esbl_status']
        
    return sorted(set(continuous_candidates + categorical_candidates + clinically_important_predictors))


In [ ]:
def filter_or_plot_df(results_df, max_ci_upper=20):
    """Filter OR results to finite and visually stable estimates.

    Parameters
    ----------
    results_df : pandas.DataFrame
        Logistic regression results containing OR and confidence interval columns.
    max_ci_upper : float, default=20
        Remove rows with upper confidence interval above this value to avoid
        unreadable plots from unstable estimates.

    Returns
    -------
    pandas.DataFrame
        Filtered plotting dataframe.
    """
    plot_df = results_df.copy()

    plot_df = plot_df[
        np.isfinite(plot_df["odds_ratio"]) &
        np.isfinite(plot_df["ci_lower"]) &
        np.isfinite(plot_df["ci_upper"])
    ]

    plot_df = plot_df[
        (plot_df["ci_lower"] > 0) &
        (plot_df["ci_upper"] < max_ci_upper)
    ].copy()

    return plot_df


def plot_univariable_or(
    cat_results_df,
    var,
    title=None,
    label_col="category",
    figsize=(8, 5),
    max_ci_upper=20,
    annotate_significant=True,
):
    """Plot univariable odds ratios for one categorical predictor.

    Parameters
    ----------
    cat_results_df : pandas.DataFrame
        Categorical univariable LR results.
    var : str
        Predictor to plot.
    title : str, optional
        Plot title. If None, a default title is used.
    label_col : str, default="category"
        Column used for y-axis labels.
    figsize : tuple, default=(8, 5)
        Figure size.
    max_ci_upper : float, default=20
        Remove very wide confidence intervals from the plotted dataframe.
    annotate_significant : bool, default=True
        Add OR text labels for rows with p < 0.05.

    Returns
    -------
    matplotlib.axes.Axes
        The plot axis.
    """
    plot_df = filter_or_plot_df(cat_results_df, max_ci_upper=max_ci_upper)

    df_var = plot_df[plot_df["predictor"] == var].copy()
    df_var = df_var.sort_values("odds_ratio")
    df_var["label"] = df_var[label_col]
    df_var["significant"] = df_var["p_value"] < 0.05

    y_pos = np.arange(len(df_var))
    fig, ax = plt.subplots(figsize=figsize)

    df_nonsig = df_var[~df_var["significant"]]
    y_nonsig = y_pos[~df_var["significant"].values]

    ax.errorbar(
        df_nonsig["odds_ratio"],
        y_nonsig,
        xerr=[
            df_nonsig["odds_ratio"] - df_nonsig["ci_lower"],
            df_nonsig["ci_upper"] - df_nonsig["odds_ratio"],
        ],
        fmt="o",
        color="gray",
        ecolor="gray",
        elinewidth=1.5,
        capsize=3,
        markersize=6,
        alpha=0.8,
        label="p ≥ 0.05",
    )

    df_sig = df_var[df_var["significant"]]
    y_sig = y_pos[df_var["significant"].values]

    ax.errorbar(
        df_sig["odds_ratio"],
        y_sig,
        xerr=[
            df_sig["odds_ratio"] - df_sig["ci_lower"],
            df_sig["ci_upper"] - df_sig["odds_ratio"],
        ],
        fmt="o",
        color="navy",
        ecolor="navy",
        elinewidth=1.5,
        capsize=3,
        markersize=6,
        label="p < 0.05",
    )

    if annotate_significant:
        for x, y, or_val in zip(df_sig["odds_ratio"], y_sig, df_sig["odds_ratio"]):
            ax.text(
                x,
                y + 0.15,
                f"OR {or_val:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
                color="navy",
            )

    ax.axvline(1, color="black", linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xticks([0.25, 0.5, 1, 2, 4])
    ax.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
    ax.ticklabel_format(style="plain", axis="x")

    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_var["label"])
    ax.set_xlabel("Odds ratio (log scale)")

    if title is None:
        title = f"Association between {var} and ESBL infection"
    ax.set_title(title, pad=12)

    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.grid(axis="y", visible=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False)

    plt.tight_layout()
    plt.show()

    return ax


In [ ]:
def plot_combined_univariable_or(
    cat_results_df,
    predictor_map=None,
    category_map=None,
    group_order=None,
    max_ci_upper=20,
):
    """Create a combined forest plot of selected categorical univariable ORs.

    Parameters
    ----------
    cat_results_df : pandas.DataFrame
        Categorical univariable LR results.
    predictor_map : dict, optional
        Mapping from raw predictor names to publication-friendly group names.
    category_map : dict, optional
        Mapping from raw category labels to publication-friendly labels.
    group_order : list[str], optional
        Display order for predictor groups.
    max_ci_upper : float, default=20
        Remove very wide confidence intervals from the plotted dataframe.

    Returns
    -------
    matplotlib.axes.Axes
        The plot axis.
    """
    if predictor_map is None:
        predictor_map = {
            "organism_bug": "Organism",
            "site": "Infection site",
            "prophylaxis_group": "Prophylaxis",
            "tfc": "Surgical specialty",
        }

    if category_map is None:
        category_map = {
            "escherichia coli": "E. coli",
            "klebsiella pneumoniae": "K. pneumoniae",
            "klebsiella oxytoca": "K. oxytoca",
            "proteus mirabilis": "P. mirabilis",
            "blood": "Blood",
            "drain": "Drain",
            "Respiratory_pleural": "Resp./pleural",
            "wound": "Wound",
            "Catheter_tips_devices": "Catheter/device",
            "Tissue/biopsy": "Tissue/biopsy",
            "Other_unknown": "Other/unknown",
            "sputum": "Sputum",
            "Genital": "Genital",
            "urine": "Urine",
            "cardiology/ cardiothoracic surgery": "Cardiothoracic",
            "hepatobiliary & pancreatic surgery": "HPB surgery",
            "trauma & orthopaedics": "T&O",
            "gynaecological oncology": "Gynae oncology",
            "plastic surgery": "Plastic",
            "general surgery": "General surgery",
            "colorectal surgery": "Colorectal",
            "vascular surgery": "Vascular",
            "neurosurgery": "Neurosurgery",
            "obstetrics": "Obstetrics",
            "urology": "Urology",
            "gynaecology": "Gynaecology",
            "ent": "ENT",
            "other": "Other",
            "nephrology": "Nephrology",
            "broad_spectrum_other": "Broad-spectrum other",
            "no_recorded_prophylaxis": "No prophylaxis recorded",
            "co-amoxiclav": "Co-amoxiclav",
            "glycopeptide": "Glycopeptide",
            "clindamycin": "Clindamycin",
            "gentamicin": "Gentamicin",
            "cefuroxime": "Cefuroxime",
        }

    if group_order is None:
        group_order = ["Organism", "Infection site", "Prophylaxis", "Surgical specialty"]

    plot_df = filter_or_plot_df(cat_results_df, max_ci_upper=max_ci_upper)
    plot_df = plot_df[plot_df["predictor"].isin(predictor_map.keys())].copy()
    plot_df["group"] = plot_df["predictor"].map(predictor_map)
    plot_df["label"] = plot_df["category"].replace(category_map)

    plot_df["group"] = pd.Categorical(plot_df["group"], categories=group_order, ordered=True)
    plot_df = plot_df.sort_values(["group", "odds_ratio"], ascending=[True, False]).reset_index(drop=True)

    y_positions = []
    current_y = 0
    for grp in group_order:
        df_grp = plot_df[plot_df["group"] == grp]
        for _ in range(len(df_grp)):
            y_positions.append(current_y)
            current_y += 1
        current_y += 1

    plot_df["y"] = y_positions
    plot_df["significant"] = plot_df["p_value"] < 0.05

    fig, ax = plt.subplots(figsize=(10, max(12, 0.45 * len(plot_df))))

    df_nonsig = plot_df[~plot_df["significant"]]
    ax.errorbar(
        df_nonsig["odds_ratio"],
        df_nonsig["y"],
        xerr=[
            df_nonsig["odds_ratio"] - df_nonsig["ci_lower"],
            df_nonsig["ci_upper"] - df_nonsig["odds_ratio"],
        ],
        fmt="o",
        color="0.65",
        ecolor="0.65",
        elinewidth=2,
        capsize=3,
        capthick=1,
        markersize=8,
        alpha=0.9,
    )

    df_sig = plot_df[plot_df["significant"]]
    ax.errorbar(
        df_sig["odds_ratio"],
        df_sig["y"],
        xerr=[
            df_sig["odds_ratio"] - df_sig["ci_lower"],
            df_sig["ci_upper"] - df_sig["odds_ratio"],
        ],
        fmt="o",
        color="navy",
        ecolor="navy",
        elinewidth=2,
        capsize=3,
        capthick=1,
        markersize=8,
    )

    for _, row in df_sig.iterrows():
        ax.text(
            row["odds_ratio"],
            row["y"] - 0.25,
            f"{row['odds_ratio']:.2f}",
            ha="center",
            va="bottom",
            fontsize=12,
            color="navy",
        )

    ax.axvline(1, color="black", linestyle="--", linewidth=2.5)

    ax.set_yticks(plot_df["y"])
    ax.set_yticklabels(plot_df["label"], fontsize=16)
    ax.set_xscale("log")
    ax.set_xticks([0.25, 0.5, 1, 2, 4])
    ax.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
    ax.tick_params(axis="x", labelsize=15)
    ax.set_xlabel("Odds ratio (95% CI, log scale)", fontsize=18, labelpad=10)
    ax.set_title("Factors associated with ESBL infection", fontsize=22, weight="bold", pad=20)

    ax.grid(axis="x", linestyle=":", alpha=0.35, linewidth=1.2)
    ax.grid(axis="y", visible=False)
    ax.spines["bottom"].set_linewidth(1.5)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.invert_yaxis()

    plt.tight_layout()
    plt.show()

    return ax


In [ ]:
def make_prophylaxis_crosstab(
    analysis_df,
    prophylaxis_col="prophylaxis_group",
    surgery_col="tfc",
    exclude_values=None,
):
    """Create prophylaxis by surgical specialty crosstabs.

    Parameters
    ----------
    analysis_df : pandas.DataFrame
        Main analysis dataframe.
    prophylaxis_col : str, default="prophylaxis_group"
        Column containing prophylaxis group/class.
    surgery_col : str, default="tfc"
        Column containing surgical specialty.
    exclude_values : list[str], optional
        Values to exclude from the prophylaxis column, e.g. ["other"].

    Returns
    -------
    tuple[pandas.DataFrame, pandas.DataFrame, pandas.DataFrame]
        Count crosstab, column-normalised crosstab, and ordered normalised crosstab.
    """
    df = analysis_df[[prophylaxis_col, surgery_col]].copy()

    if exclude_values is not None:
        df = df[~df[prophylaxis_col].isin(exclude_values)]

    ct = pd.crosstab(df[prophylaxis_col], df[surgery_col])
    ct_norm = ct.div(ct.sum(axis=0), axis=1)

    row_order = ct.sum(axis=1).sort_values(ascending=False).index
    col_order = ct.sum(axis=0).sort_values(ascending=False).index
    ct_norm_ord = ct_norm.loc[row_order, col_order]

    return ct, ct_norm, ct_norm_ord


def plot_prophylaxis_heatmap(ct_norm_ord, title="Chosen prophylaxis class per surgery speciality", figsize=(12, 8)):
    """Plot a heatmap of column-normalised prophylaxis use by surgical specialty.

    Parameters
    ----------
    ct_norm_ord : pandas.DataFrame
        Ordered column-normalised crosstab.
    title : str
        Plot title.
    figsize : tuple
        Figure size.

    Returns
    -------
    matplotlib.axes.Axes
        The plot axis.
    """
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(ct_norm_ord, cmap="viridis", ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return ax


def plot_prophylaxis_heatmap_panel(ct, ct_norm, ct_norm_ord, top_n=8, figsize=(12, 10)):
    """Plot the four-panel heatmap diagnostics from the original notebook.

    Parameters
    ----------
    ct : pandas.DataFrame
        Count crosstab.
    ct_norm : pandas.DataFrame
        Column-normalised crosstab.
    ct_norm_ord : pandas.DataFrame
        Ordered column-normalised crosstab.
    top_n : int, default=8
        Number of most common prophylaxis groups to show in the bottom-left
        and bottom-right plots.
    figsize : tuple, default=(12, 10)
        Figure size.

    Returns
    -------
    numpy.ndarray
        Array of matplotlib axes.
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)

    sns.heatmap(ct_norm, ax=axes[0, 0], cmap="viridis")
    axes[0, 0].set_title("Prophylaxis per surgical specialty (normalised)")

    sns.heatmap(np.log1p(ct_norm), ax=axes[0, 1], cmap="viridis")
    axes[0, 1].set_title("Prophylaxis per surgical specialty (log scale)")

    top_abx = ct.sum(axis=1).nlargest(top_n).index
    sns.heatmap(ct_norm_ord.loc[top_abx], ax=axes[1, 0], cmap="viridis")
    axes[1, 0].set_title(f"Prophylaxis per surgical specialty (top {top_n} abx)")

    sns.heatmap(np.log1p(ct_norm_ord.loc[top_abx]), ax=axes[1, 1], cmap="viridis")
    axes[1, 1].set_title("Log-scaled ordered")

    plt.tight_layout()
    plt.show()

    return axes


def plot_prophylaxis_stacked_bar(ct_norm, cluster_columns=True, figsize=(14, 6)):
    """Plot stacked bars for prophylaxis groups across surgical specialties.

    Parameters
    ----------
    ct_norm : pandas.DataFrame
        Column-normalised crosstab.
    cluster_columns : bool, default=True
        If True, order surgical specialties using hierarchical clustering.
        If False, order by total counts in the crosstab.
    figsize : tuple, default=(14, 6)
        Figure size.

    Returns
    -------
    matplotlib.axes.Axes
        The plot axis.
    """
    if cluster_columns:
        Z = linkage(ct_norm.T, method="ward")
        col_order = ct_norm.columns[leaves_list(Z)]
    else:
        col_order = ct_norm.sum(axis=0).sort_values(ascending=False).index

    row_order = ct_norm.sum(axis=1).sort_values(ascending=False).index
    ct_norm_ord = ct_norm.loc[row_order, col_order]

    colors = sns.color_palette("Set2", n_colors=max(3, len(ct_norm_ord.index)))
    ax = ct_norm_ord.T.plot(kind="bar", stacked=True, figsize=figsize, color=colors)

    ax.set_title("Distribution of prophylaxis groups across surgical specialties")
    ax.set_xlabel("Surgical specialty")
    ax.set_ylabel("Proportion")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.legend(title="Prophylaxis group", bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()
    plt.show()

    return ax


## Run analysis

In [ ]:
analysis_df = load_analysis_data("../data/interim/analysis_df.csv", surgery_only=True)

In [ ]:
analysis_df.columns

In [ ]:
analysis_df[(analysis_df['n_sites'] > 1) & (~analysis_df['all_sites_grouped'].str.contains('other')) ][['all_sites_grouped', 'site']].value_counts().reset_index()

In [ ]:
analysis_df.columns

In [ ]:
def regroup_prophylaxis(x):
    if pd.isna(x):
        return pd.NA
    
    x = x.lower().strip()

    if "cefuroxime" in x and "metronidazole" in x:
        return "cefuroxime + metronidazole"
    
    if "co-amoxiclav" in x:
        return "co-amoxiclav-based"
    
    if "glycopeptide" in x or "teicoplanin" in x or "vancomycin" in x:
        return "glycopeptide-based"
    
    if "aminoglycoside" in x or "gentamicin" in x:
        return "aminoglycoside-based"
    
    if "clindamycin" in x:
        return "clindamycin-based"
    
    if "metronidazole" in x:
        return "metronidazole-only/+other"
    
    if "cefuroxime" in x:
        return "cefuroxime-only/+other"
    
    if "piperacillin-tazobactam" in x:
        return "broad-spectrum beta-lactam"
    
    return "other"

analysis_df["prophylaxis_group"] = (
    analysis_df["prophylaxis_group"]
    .apply(regroup_prophylaxis))


In [ ]:
analysis_df['ethnicity_desc']=analysis_df['ethnicity_desc'].replace({
        'mixed - any other mixed background': 'mixed',
        'mixed - white and black caribbean' :'mixed',
        'mixed - white and asian': 'mixed',
        'mixed - white and black african' : 'mixed', 
        'mixed' : 'mixed'})

analysis_df = analysis_df.drop(columns ='prophylaxis_class', errors = 'ignore')

In [ ]:
analysis_df['gender'] = analysis_df['gender'].map({2: 1, 1: 0})

In [ ]:
analysis_df.groupby('esbl_status')['in_hosp_mortality'].agg(mean = 'mean', std = 'std', rates = (lambda x : round(100*x.mean(),1)))

### Table 1

In [ ]:
table1_cols = cfg["table1"]

table1_df = get_table1_df(analysis_df, table1_cols)
table1 = create_table1(table1_df, table1_cols, export_path="table1_epidemiology_esbl.tex")

print(table1)


In [ ]:
missing = analysis_df.isna().sum().sort_values(ascending=False)
missing

### Univariable logistic regression

In [ ]:
analysis_df['ethnicity_desc']= analysis_df['ethnicity_desc'].replace({'mixed':pd.NA})

In [ ]:
uni_lr = cfg["univariable_lr"]

uni_lr_df = prepare_lr_df(analysis_df, uni_lr)

continuous_vars = uni_lr["continuous"]
categorical_vars = uni_lr["categorical"]
outcome = uni_lr["outcome"]

reference_levels = {
    "site": "blood",
    "organism_bug": "escherichia coli",
    "ethnicity_desc": "white",
    "tfc": "general_surgery",
    "prophylaxis_group": "cefuroxime + metronidazole",
}

results_df = run_univariable_continuous_lr(
    uni_lr_df,
    continuous_vars=continuous_vars,
    outcome=outcome,
)

cat_results_df = run_univariable_categorical_lr(
    uni_lr_df,
    categorical_vars=categorical_vars,
    outcome=outcome,
    reference_levels=reference_levels,
)

variables_for_multi_lr = select_variables_for_multivariable_lr(
    results_df,
    cat_results_df,
    p_threshold=0.2,
)

uni_lr_df.to_csv("lr_data.csv", index=False)
results_df.to_csv("univariable_lr_continuous_results.csv", index=False)
cat_results_df.to_csv("univariable_lr_categorical_results.csv", index=False)

variables_for_multi_lr


In [ ]:
first_episode_df = (
    analysis_df
    .sort_values(["subject", "infection_id"])
    .drop_duplicates("subject", keep="first")
)

first_episode_df = prepare_lr_df(first_episode_df, uni_lr)
uni_lr_df.to_csv("first_ep_data.csv", index=False)

In [ ]:
results_df

In [ ]:
cat_results_df

In [ ]:
# View significant univariable associations
display(results_df[results_df["p_value"] < 0.05].sort_values("p_value"))
display(cat_results_df[cat_results_df["p_value"] < 0.2].sort_values("p_value"))


### Plot univariable regression ORs

In [ ]:
plot_univariable_or(
    cat_results_df,
    var="tfc",
    title="Association between surgical specialty and ESBL infection",
    figsize=(8, 5),
)

plot_univariable_or(
    cat_results_df,
    var="ethnicity_desc",
    title="Association between ethnicity and ESBL infection",
    figsize=(8, 5),
)

plot_univariable_or(
    cat_results_df,
    var="site",
    title="Association between sample site and ESBL infection",
    figsize=(8, 4),
)

plot_univariable_or(
    cat_results_df,
    var="prophylaxis_group",
    title="Association between prophylaxis and ESBL infection",
    figsize=(8, 4),
)


In [ ]:
plot_combined_univariable_or(cat_results_df)


## Multivariable regression

TODO: once variables are clinically/statistically selected, fit the final multivariable logistic regression here. Avoid choosing variables only by p-value; include key confounders based on the causal question and clinical reasoning.

In [ ]:
def run_multivariable_lr(uni_lr_df, variables, outcome="esbl_status", categorical_vars=None, reference_levels=None):
    """Fit a multivariable logistic regression model.

    Parameters
    ----------
    uni_lr_df : pandas.DataFrame
        Logistic regression dataframe.
    variables : list[str]
        Predictors to include in the model.
    outcome : str, default="esbl_status"
        Binary outcome variable coded 1/0.
    categorical_vars : list[str], optional
        Variables that should be wrapped in C(...).
    reference_levels : dict, optional
        Mapping of categorical variable to desired reference category.

    Returns
    -------
    tuple[statsmodels.discrete.discrete_model.BinaryResultsWrapper, pandas.DataFrame]
        Fitted model and tidy OR table.
    """
    if categorical_vars is None:
        categorical_vars = []
    if reference_levels is None:
        reference_levels = {}

    terms = []
    for var in variables:
        if var in categorical_vars:
            if var in reference_levels:
                terms.append(f"C({var}, Treatment(reference={reference_levels[var]!r}))")
            else:
                terms.append(f"C({var})")
        else:
            terms.append(var)

    formula = f"{outcome} ~ " + " + ".join(terms)
    model_df = uni_lr_df[[outcome] + variables].dropna()
    model = smf.logit(formula, data=model_df).fit(disp=0)

    conf = model.conf_int()
    tidy = pd.DataFrame({
        "term": model.params.index,
        "odds_ratio": np.exp(model.params),
        "ci_lower": np.exp(conf[0]),
        "ci_upper": np.exp(conf[1]),
        "p_value": model.pvalues,
    }).reset_index(drop=True)

    tidy = tidy[tidy["term"] != "Intercept"].copy()
    tidy["n"] = len(model_df)

    return model, tidy


# Example only: edit this list before using as the final adjusted model.
# final_variables = variables_for_multi_lr
# multi_model, multi_results_df = run_multivariable_lr(
#     uni_lr_df,
#     variables=final_variables,
#     outcome=outcome,
#     categorical_vars=categorical_vars,
#     reference_levels=reference_levels,
# )
# display(multi_results_df.sort_values("p_value"))


In [ ]:
cat_results_df

## Plotting prophylaxis per surgical specialty

In [ ]:
ct_prophylaxis_class, ct_prophylaxis_class_norm, ct_prophylaxis_class_norm_ord = make_prophylaxis_crosstab(
    analysis_df,
    prophylaxis_col="prophylaxis_class",
    surgery_col="tfc",
)

plot_prophylaxis_heatmap(
    ct_prophylaxis_class_norm_ord,
    title="Chosen prophylaxis class per surgery speciality",
)


In [ ]:
ct, ct_norm, ct_norm_ord = make_prophylaxis_crosstab(
    analysis_df,
    prophylaxis_col="prophylaxis_group",
    surgery_col="tfc",
    exclude_values=["other"],
)

plot_prophylaxis_heatmap_panel(ct, ct_norm, ct_norm_ord, top_n=8)
plot_prophylaxis_stacked_bar(ct_norm, cluster_columns=True)


## Outputs to prepare for the epidemiology publication

For the paper, I would aim to produce the following outputs.

### Core descriptive outputs

1. **Cohort flow diagram**
   - Number of admissions/patients initially eligible.
   - Number excluded at each step.
   - Final number of surgical patients with infection.
   - Final ESBL and non-ESBL counts.

2. **Table 1: baseline characteristics**
   - Stratified by ESBL vs non-ESBL.
   - Include missingness.
   - Present continuous variables as median (IQR) if non-normal.
   - Present categorical variables as n (%).
   - Avoid over-interpreting p-values in Table 1; use it mainly descriptively.

3. **Microbiology summary**
   - Organism distribution overall and by ESBL status.
   - Infection/sample site distribution overall and by ESBL status.
   - Time from admission/surgery to infection, if relevant.

4. **Antibiotic prophylaxis summary**
   - Prophylaxis group/class distribution overall and by surgical specialty.
   - Prophylaxis group/class distribution by ESBL status.
   - Consider whether “no recorded prophylaxis” means truly none or missing documentation.

### Regression outputs

5. **Univariable logistic regression table**
   - Predictor, reference category, OR, 95% CI, p-value, and denominator used.
   - For categorical variables, include all levels and clearly state reference categories.

6. **Multivariable logistic regression table**
   - Adjusted OR, 95% CI, p-value.
   - Include the final adjustment set and justify it clinically/epidemiologically.
   - Avoid relying only on p < 0.2 screening; also include known confounders.

7. **Forest plot**
   - A clean forest plot of the final adjusted model is more publication-relevant than many separate univariable plots.
   - Keep the univariable forest plot as exploratory or supplementary.

### Model diagnostics / robustness

8. **Missing data summary**
   - Missingness per candidate predictor.
   - Complete-case sample size for each model.
   - Consider whether missingness is differential by ESBL status.

9. **Sparse category checks**
   - Counts and ESBL events per category before modelling.
   - Collapse rare categories before regression where clinically sensible.
   - Flag unstable ORs with very wide CIs.

10. **Collinearity / confounding checks**
   - Check overlap between surgical specialty, prophylaxis, sample site, organism, and hospital pathway.
   - Prophylaxis and surgical specialty may be highly linked, so interpret both carefully if included together.

11. **Sensitivity analyses**
   - Repeat models excluding “no recorded prophylaxis” if this may represent missingness.
   - Repeat with broader collapsed prophylaxis groups/classes.
   - Consider patient-level clustering if patients can appear more than once.
   - Consider excluding likely colonisation/contamination if clinically relevant.

### Things to clarify before writing results

- Exact unit of analysis: patient, admission, infection episode, or microbiology sample.
- Whether repeated infections/samples per patient exist and how they are handled.
- Definition of ESBL status and how intermediate/resistant susceptibility results were coded.
- Definition of surgical exposure and infection timing.
- How missing/unknown values are treated.
- Which variables are pre-infection predictors versus post-infection descriptors.
